# Trabajo Práctico  - Calibración y Pose de la Cámara

## Resolución

## GRUPO: NOMBRE DEL GRUPO

# Integrantes
* integrante
* integrante

In [ ]:
import os
import importlib
import numpy as np
import alc

importlib.reload(alc)  # los cambios en alc.py no se ven sin reload
np.set_printoptions(precision=4, suppress=True)


### Ejercicio 1 - DLT ecuaciones formales

Siendo $Q\in \mathbb{R}^{2n\times 12}$, ¿qué rango debe tener para asegurar una solución no nula única a menos de un factor de escala? ¿Cuántos puntos, como mínimo, aseguran la solución y qué condición deben cumplir?

Encontrar el vector $\mathbf{m}$ que resuelve la minimización de la ecuación 6.

### Ejercicio 2 - Calibración con puntos 3D

a) Utilizando los datos de la carpeta $\texttt{calib/3d}$ calcular $\mathbf{R}$,  $\mathbf{t}$ y $\mathbf{K}$.

Calcular un nuevo set de parámetros para 

b) $\alpha_u = \alpha_v$ 

c) $K_{12}=0$



In [ ]:
# Ej 2-a: calibración con puntos 3D (Tsai/DLT) — datos de calib/3d
objPts = np.loadtxt(os.path.join("calib", "3d", "p_W_corners.txt"),
                    delimiter=",")   # (12, 3) en cm, NO coplanarios
detected = np.loadtxt(os.path.join("calib", "3d",
                                   "detected_corners.txt"))
# (210, 24): 1 fila por vista con las 12 esquinas intercaladas (u, v)
N_PTS, N_VISTAS = objPts.shape[0], detected.shape[0]

Ks, Rs, ts = [], [], []
err_repro = np.zeros(N_VISTAS)
for k in range(N_VISTAS):
    imgPts = detected[k].reshape(N_PTS, 2)   # (u, v) de esta vista
    Q = alc.obtenerQ3d(imgPts, objPts)       # (2n, 12): sistema ec. 6
    M = alc.resolverDLT3D(Q)                 # μ·K[R|t], con m34 >= 0
    Rj, tj, Kj = alc.devolverMatricesCalibracion3d(M)
    Ks.append(Kj); Rs.append(Rj); ts.append(tj)
    # QA por vista: error de reproyección con la M estimada
    err_max = 0.0
    for i in range(N_PTS):
        P = np.append(objPts[i], 1.0)
        w = alc.multiplicar(M, P)
        err_max = max(err_max, abs(w[0]/w[2] - imgPts[i, 0]),
                                abs(w[1]/w[2] - imgPts[i, 1]))
    err_repro[k] = err_max

# Intrínsecos: promedio sobre las vistas (la cámara es la misma)
K = np.mean(Ks, axis=0)
print("K promedio:\n", K)
print("std por vista de (αu, γ, u0):", np.std(Ks, axis=0)[0].round(2))
print("std por vista de (αv, v0):", np.std(Ks, axis=0)[1].round(2))
print("err reproyección: max %.2f px | promedio %.2f px"
      % (err_repro.max(), err_repro.mean()))

# Pose (R, t): escena estática → promedio entre vistas.
# El promedio lineal de rotaciones NO es rotación exacta: se
# re-ortonormaliza con el helper de alc.py (Gram-Schmidt corto).
Rm = np.mean(Rs, axis=0)
R = alc._ortonormalizarR(Rm[0], Rm[1], Rm[2])
t = np.mean(ts, axis=0)
print("R (promedio, re-ortonormal):\n", R)
print("t promedio (cm):", t)


K promedio:
 [[425.9314  -0.3365 357.0923]
 [  0.     423.5597 254.0073]
 [  0.       0.       1.    ]]
std por vista de (αu, γ, u0): [4.81 1.11 2.43]
std por vista de (αv, v0): [0.   4.66 3.51]
err reproyección: max 1.36 px | promedio 0.84 px
R (promedio, re-ortonormal):
 [[ 0.5471  0.1272 -0.8273]
 [-0.3369  0.9382 -0.0785]
 [ 0.7663  0.3217  0.5562]]
t promedio (cm): [-17.98   -13.8492  39.083 ]


**Resultados Ej 2-a**

Con las $210$ vistas del video y los $12$ puntos de la mira (que **no**
son coplanarios: su $Z$ toma los valores $0$, $-10.95$ y $-24.45$ cm; con
$n=12 \geq 6$ el rango $11$ de $Q$ se alcanza, cf. Ej 1-a), el pipeline
DLT 3D + descomposición recupera:

$$K_{\text{prom}} = \begin{pmatrix} 425.93 & -0.34 & 357.09 \\
0 & 423.56 & 254.01 \\ 0 & 0 & 1 \end{pmatrix}$$

- **Plausibilidad**: la resolución es $752\times 480$ (centro esperado
  $(376, 240)$), y el despeje da $u_0 = 357.1$, $v_0 = 254.0$: razonables.
  Con $f \approx 424$ px el FOV horizontal es $\approx 83^\circ$ (cámara
  ancha, coherente con un webcam/videollamada).
- **Estabilidad**: el desvío estándar por vista es $\sim 5$ px en los
  focales y $\sim 3$ px en el centro (escena estática; la dispersión
  proviene del ruido de detección de esquinas).
- **Error de reproyecci?n**: m?ximo $1.36$ px, promedio $0.84$ px
  sobre las $210$ vistas ? del mismo orden que el ruido de
  detecci?n ($\sim 0.5$ px por esquina): no hay error sistem?tico.
- **Skew**: $\gamma = K_{12} \approx -0.3$ px, prácticamente nulo; es el
  gancho natural para las imposiciones de b) y c).


In [ ]:
# Ej 2-b/c: nuevo set de parámetros imponiendo las restricciones del
# enunciado (d): αu = αv y K12 = 0, promediadas sobre las mismas vistas.
K_b = np.zeros((3, 3))
K_c = np.zeros((3, 3))
for k in range(N_VISTAS):
    imgPts = detected[k].reshape(N_PTS, 2)
    M = alc.resolverDLT3D(alc.obtenerQ3d(imgPts, objPts))
    Kb = alc.devolverMatricesCalibracion3d(M, au_av=True)[2]
    Kc = alc.devolverMatricesCalibracion3d(M, gamma_0=True)[2]
    K_b = K_b + Kb
    K_c = K_c + Kc
K_b = K_b / N_VISTAS
K_c = K_c / N_VISTAS
print("K_b (αu = αv):\n", K_b)
print("K_c (K12 = 0):\n", K_c)


K_b (αu = αv):
 [[423.5597  -0.3365 357.0923]
 [  0.     423.5597 254.0073]
 [  0.       0.       1.    ]]
K_c (K12 = 0):
 [[425.933    0.     357.0923]
 [  0.     423.5597 254.0073]
 [  0.       0.       1.    ]]


**Comparación de los tres sets de parámetros**

$$K_\text{base} = \begin{pmatrix} 425.9 & -0.3 & 357.1 \\
0 & 423.6 & 254.0 \\ 0 & 0 & 1 \end{pmatrix},
\quad
K_b = \begin{pmatrix} 423.6 & -0.3 & 357.1 \\
0 & 423.6 & 254.0 \\ 0 & 0 & 1 \end{pmatrix},
\quad
K_c = \begin{pmatrix} 425.9 & 0 & 357.1 \\
0 & 423.6 & 254.0 \\ 0 & 0 & 1 \end{pmatrix}$$

Como $\gamma$ de partida es casi nulo ($\approx -0.3$ px) y
$\alpha_u \approx \alpha_v$ ($\sim 0.5\%$ de diferencia), las
imposiciones cambian los parámetros por cantidades del orden del ruido
de detección — consistente con que ambas restricciones son
razonables para esta cámara.


### Ejercicio 3 - Calibración con puntos coplanares

Resolver la calibración para el caso de puntos coplanares, trabajando sobre los datos de la carpeta $\texttt{calib/coplanar}$ con las imágenes $\texttt{Im\_R\_1.png}$ a $\texttt{Im\_R\_20.png}$.


a) Obtener la matriz $\mathbf{K}$

b) Encontrar la pose de $\texttt{Im\_R\_1.png}$, $\texttt{Im\_R\_10.png}$ y $\texttt{Im\_R\_20.png}$.

### Ejercicio 4 - Fútbol

a) Calcular la homografía de la cancha de futbol de la foto de $\texttt{gol-enzo.png}$. Encontrar la distancia a la que se encontraba Enzo. 
    Dibujar sobre la fotografía círculos de 1 metro de diámetro alrededor de los jugadores argentinos.

b) Calcular la homografía de la changa de fútbol de la foto $\texttt{posibleoffside.jpg}$. 
    Trazar sobre la fotografía una recta sobre el pié del jugador del equipo defensor que defina si hay fuera de juego.
    

### Ejercicio 5 - Cruzando en rojo - Cálculo de velocidad

a) Calcular los parámetros intrínsecos de la cámara a partir de las diferentes vistas de la esquina y las coordenadas de los puntos en el archivo $\texttt{world\_img\_pts.csv}$

b) A partir de los puntos $\texttt{gs\_10\_09\_point\_correspondences\_v5.txt}$ calcular la pose de la cámara y reproyectar los puntos sobre la imagen $\texttt{img\_10\_18\_47\_351.jpg}$.

c) Calcular la velocidad de los dos primeros vehículos (taxi y camioneta) y de la señora de buzo rojo que cruza la calle. Graficar las velocidades en un plot temporal a lo largo de la secuencia.

### Ejercicio 6 - Aterrizaje suave

a) Aplicar POSIT para encontrar la pose ($\mathbf{R}$ y $\mathbf{t}$) de la cámara del Drone a partir de la imagen $\texttt{row\_00248.png}$. La cámara tiene la matriz intrínseca $\mathbf{K}$. 

b) Estimar la matriz de rotación y traslación para ubicar al drone exactamente sobre la letra H y que la cámara la vea vertical (el punto blanco marca la parte superior izquierda)

---